# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
1. [1. 2. 3.] и [0 1 1] в первом случае каждое число делится на 2 обычным способом во втором при делении на 4 остаётся только целая часть

In [ ]:
2. [4. 4. 4.] в массиве три строки и в каждой четыре единицы сумма каждой строки равна 4

In [ ]:
3. [[0 1 2], [1 2 3], [2 3 4]] числа 0, 1, 2 прибавляются к каждому числу в отдельной строке

In [ ]:
4. 6 корни из 1, 4 и 9 равны 1, 2 и 3 их сумма  6

In [ ]:
5. 7.5 квадраты чисел 1, 2, 3, 4 это 1, 4, 9, 16 их среднее равно 7.5

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1

In [4]:
month = np.array([r["order_date"][5:7] for r in rows])

In [5]:
september = clean & (month == "09")
october = clean & (month == "10")

In [6]:
sep_revenue = amount[september].sum()
oct_revenue = amount[october].sum()
change = (oct_revenue - sep_revenue) / sep_revenue * 100

In [7]:
clean.sum()

np.int64(4739)

In [9]:
print(clean.sum())

4739


In [10]:
sep_revenue

np.float64(35243410.0)

In [11]:
print(sep_revenue)

35243410.0


In [12]:
oct_revenue

np.float64(34419520.0)

In [13]:
print(oct_revenue)

34419520.0


In [14]:
round(change, 1)

np.float64(-2.3)

In [15]:
print(round(change, 1), "%")

-2.3 %


совпадает с результатом дз 2

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2

In [16]:
categories = np.unique(cat)

In [17]:
category_mask = categories[:, None] == cat[None, :]


In [18]:
september = amount * clean * (month == "09")
october = amount * clean * (month == "10")

In [19]:
sep_by_category = (category_mask * september).sum(axis=1)
oct_by_category = (category_mask * october).sum(axis=1)

In [20]:
table = np.column_stack((sep_by_category, oct_by_category))

In [21]:
categories

array(['Дом', 'Книги', 'Красота', 'Одежда', 'Электроника'], dtype='<U11')

In [22]:
print(categories)

['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']


In [23]:
table.astype(int)

array([[ 4997730,  5125790],
       [ 1065300,  1140910],
       [ 2610950,  2678990],
       [ 9167100,  8505730],
       [17402330, 16968100]])

In [24]:
print(table.astype(int))

[[ 4997730  5125790]
 [ 1065300  1140910]
 [ 2610950  2678990]
 [ 9167100  8505730]
 [17402330 16968100]]


электроника принесла 34370430 за два месяца больше остальных категорий
Сильнее всего упала одежда на 7,2% относительно сентября
axis=1 складывает сентябрь и октябрь внутри каждой строки категории

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3

In [25]:
categories = np.unique(cat)
category_mask = categories[:, None] == cat[None, :]

In [26]:
september_count = (category_mask & clean & (month == "09")).sum(axis=1)
october_count = (category_mask & clean & (month == "10")).sum(axis=1)

In [27]:
count_table = np.column_stack((september_count, october_count))
average_table = table / count_table

In [28]:
count_table

array([[476, 476],
       [265, 299],
       [422, 435],
       [678, 652],
       [531, 505]])

In [30]:
print(count_table)

[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]


In [32]:
average_table[:, 1].round()

array([10768.,  3816.,  6159., 13046., 33600.])

In [35]:
print(average_table[:, 1].round())

[10768.  3816.  6159. 13046. 33600.]


In [ ]:
categories[average_table[:, 1].argmax()]

In [36]:
print(categories[average_table[:, 1].argmax()])

Электроника


In [ ]:
электроника - средний чек около 33600  в октябре, самый высокий среди категорий

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4

In [57]:
cities = np.unique(city)
city_mask = cities[:, None] == city[None, :]

In [58]:
september = amount * clean * (month == "09")
october = amount * clean * (month == "10")

In [59]:
city_table = np.column_stack((
    (city_mask * september).sum(axis=1),
    (city_mask * october).sum(axis=1)
))

In [60]:
city_share = city_table / city_table.sum(axis=0)


In [61]:
cities

array(['Актобе', 'Алматы', 'Астана', 'Атырау', 'Караганда', 'Шымкент'],
      dtype='<U9')

In [62]:
print(cities)

['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']


In [63]:
city_table.astype(int)

array([[ 2576960,  2540420],
       [14501070, 14643020],
       [ 9034220,  8850530],
       [  923990,   969860],
       [ 3648520,  3371680],
       [ 4558650,  4044010]])

In [64]:
print(city_table.astype(int))

[[ 2576960  2540420]
 [14501070 14643020]
 [ 9034220  8850530]
 [  923990   969860]
 [ 3648520  3371680]
 [ 4558650  4044010]]


In [65]:
city_share.sum(axis=0)

array([1., 1.])

In [66]:
print(city_share.sum(axis=0))

[1. 1.]


In [67]:
shymkent = cities == "Шымкент"

In [68]:
round(city_share[shymkent, 0][0] * 100, 2)

np.float64(12.93)

In [72]:
print(round(city_share[shymkent, 0][0] * 100, 2), "%")

12.93 %


In [70]:
round(city_share[shymkent, 1][0] * 100, 2)

np.float64(11.75)

In [74]:
print(round(city_share[shymkent, 1][0] * 100, 2), "%")

11.75 %


каждый столбец в сумме даёт 1 
Доля Шымкента снизилась с 12,93% до 11,75%, то есть на 1,19 процентного пункта 
axis=0 складывает выручку всех городов отдельно за каждый месяц

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5

In [75]:
city_change = city_table[:, 1] - city_table[:, 0]
total_change = city_change.sum()

In [76]:
contribution = city_change / total_change * 100

In [77]:
print(cities)

['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']


In [78]:
contribution.round(1)

array([  4.4, -17.2,  22.3,  -5.6,  33.6,  62.5])

In [79]:
print(contribution.round(1))

[  4.4 -17.2  22.3  -5.6  33.6  62.5]


In [80]:
shymkent = cities == "Шымкент"

In [81]:
round(contribution[shymkent][0], 1)

np.float64(62.5)

In [82]:
print(round(contribution[shymkent][0], 1), "%")

62.5 %


In [ ]:
Шымкент дал 62,5% общего падения выручки
В первую очередь стоит выяснить причину снижения выручки в Шымкенте на этот город пришлось 62,5% общего падения

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6

In [83]:
share = city_table / city_table.sum(axis=1)[:, None]

In [84]:
(share * 100).round(2)

array([[50.36, 49.64],
       [49.76, 50.24],
       [50.51, 49.49],
       [48.79, 51.21],
       [51.97, 48.03],
       [52.99, 47.01]])

In [85]:
print((share * 100).round(2))

[[50.36 49.64]
 [49.76 50.24]
 [50.51 49.49]
 [48.79 51.21]
 [51.97 48.03]
 [52.99 47.01]]


In [86]:
share.sum(axis=1)

array([1., 1., 1., 1., 1., 1.])

In [87]:
print(share.sum(axis=1))

[1. 1. 1. 1. 1. 1.]


Код падает, потому что city_table имеет форму 6 * 2, а city_table.sum(axis=1) это 6 чисел
NumPy не понимает как разделить две суммы в каждой строке на соответствующий итог города

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

In [89]:
x = amount[clean]

In [90]:
z = (x - x.mean()) / x.std()

In [91]:
count = (abs(z) > 3).sum()

In [92]:
percent = count / len(x) * 100

In [93]:
count

np.int64(87)

In [95]:
print("Заказов с |z| > 3:", count)

Заказов с |z| > 3: 87


In [96]:
round(percent, 2)

np.float64(1.84)

In [97]:
print(round(percent, 2), "%")

1.84 %


In [99]:
print("Среднее z:", round(z.mean(), 10))

Среднее z: 0.0


In [101]:
print("Стандартное отклонение z:", round(z.std(), 10))

Стандартное отклонение z: 1.0


In [ ]:
среднее z должно быть близко к 0, а стандартное отклонение  к 1

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8

In [106]:
mask = clean & ~np.isnan(days)

In [107]:
X = np.column_stack((amount[mask], days[mask]))

In [108]:
scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

In [109]:
scaled.min(axis=0)

array([0., 0.])

In [110]:
print(scaled.min(axis=0))

[0. 0.]


In [111]:
scaled.max(axis=0)

array([1., 1.])

In [112]:
print(scaled.max(axis=0))

[1. 1.]


Модели нужны признаки в одном масштабе чтобы большие числа не влияли на расчёты сильнее только из-за своих единиц измерения
Например сумма заказа может быть в тысячах тенге, а доставка в днях